# Comparison of Physics-Informed Neural Networks (PINNs) and Experimental Reality in Fluid Viscosity Dynamics

**Paper:** Septiana, E., Saefan, J., Siswanto, J. (2026). *Comparison of Physics-Informed Neural Networks (PINNs) and Experimental Reality in Fluid Viscosity Dynamics.* Physics Communication, 10(1), 37-49.

**Carpeta origen:** `PINNs/1. mecanica de fluidos/Comparison of Physics-Informed Neural Networks (PINNs) and Experimental Reality in Fluid Viscosity Dynamics.pdf`

## Como se usan las PINNs en este paper

El paper usa una PINN para modelar la velocidad $v_\theta(t)$ de una esfera cayendo en glicerina (viscosimetro de esfera), comparando la prediccion contra la solucion analitica de Stokes y contra datos experimentales de 5 sensores infrarrojos. La EDO de movimiento (Eq. 9-10) es:

$$\frac{dv}{dt} = A - Bv, \qquad A=\left(1-\frac{\rho_f}{\rho_s}\right)g,\quad B=\frac{9\mu}{2\rho_s r^2}$$

con solucion analitica $v(t)=v_t(1-e^{-Bt})$, $v_t=A/B$. La red se define con una **restriccion dura de condicion inicial** $v_\theta(t)=t\cdot N_\theta(t)$ (garantiza $v_\theta(0)=0$ automaticamente, sin necesitar un termino de perdida para ello), 3 capas ocultas x 64 neuronas, activacion tanh (Tabla 3). La perdida combina el residuo fisico de la EDO (Eq. 16-17) y el error contra los datos experimentales (Eq. 18):

$$\mathcal{R}(t)=\frac{dv_\theta}{dt}-(A-Bv_\theta), \qquad \mathcal{L}_{total}=\mathcal{L}_{data}+\lambda\,\mathcal{L}_{physics},\ \ \lambda=10$$

entrenada con Adam (lr=$10^{-3}$), 5000 epocas, $N_f=2000$ puntos de colocacion en tiempo normalizado $t_{norm}\in[0,1]$ ($t_{phys}\in[0,6]$ s).

Este cuaderno reproduce fielmente esa arquitectura, restriccion dura, funcion de perdida y protocolo de entrenamiento, usando los parametros fisicos (Tabla 1) y los datos experimentales (Tabla 2) reportados en el paper.

## Repositorio publico de referencia

El PDF no incluye enlace a codigo (es un articulo de una revista de educacion en fisica, sin seccion de disponibilidad de datos/codigo). Al buscar en GitHub un repositorio publico que implemente PINNs para EDOs de primer orden con condicion inicial dura, el mas cercano y ampliamente referenciado es:

- **madagra/basic-pinn** &mdash; https://github.com/madagra/basic-pinn — implementacion basica de PINNs para EDOs/EDPs en PyTorch, mismo patron (red + diferenciacion automatica + perdida de residuo) que el usado aqui.

In [ ]:
# Instalacion de dependencias (ejecutar si no estan ya instaladas en el entorno)
%pip install -q torch numpy matplotlib

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

torch.manual_seed(0)
np.random.seed(0)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

## 1. Parametros fisicos (Tabla 1) y datos experimentales (Tabla 2)

In [ ]:
r = 0.010125     # radio de la esfera [m]
rho_s = 1297.2   # densidad de la esfera [kg/m^3]
rho_f = 1000.0   # densidad del fluido (glicerina) [kg/m^3]
mu = 0.69        # viscosidad dinamica [Pa.s]
g = 9.81         # gravedad [m/s^2]
t_max = 6.0      # tiempo maximo de observacion [s], usado para normalizar

A = (1 - rho_f / rho_s) * g
B = 9 * mu / (2 * rho_s * r**2)
v_terminal = A / B
print(f'A={A:.5f}  B={B:.5f}  v_terminal={v_terminal:.4f} m/s (paper reporta 0.096 m/s)')

# Tabla 2: datos experimentales (tiempo fisico [s], velocidad [m/s])
t_exp = np.array([0.00, 0.7696, 1.6808, 2.713, 5.1562])
v_exp = np.array([0.000, 0.1299, 0.1097, 0.0968, 0.0920])

t_exp_t = torch.tensor(t_exp / t_max, dtype=torch.float32, device=device).view(-1, 1)
v_exp_t = torch.tensor(v_exp, dtype=torch.float32, device=device).view(-1, 1)

## 2. Red PINN con restriccion dura de condicion inicial (Tabla 3): $v_\theta(t) = t \cdot N_\theta(t)$

In [ ]:
class PINN(nn.Module):
    def __init__(self, n_hidden_layers=3, n_neurons=64):
        super().__init__()
        layers = [nn.Linear(1, n_neurons), nn.Tanh()]
        for _ in range(n_hidden_layers - 1):
            layers += [nn.Linear(n_neurons, n_neurons), nn.Tanh()]
        layers += [nn.Linear(n_neurons, 1)]
        self.net = nn.Sequential(*layers)

    def forward(self, t_norm):
        # Restriccion dura: v(0)=0 se satisface automaticamente (paper, seccion 'Computation of PINNs')
        return t_norm * self.net(t_norm)


model = PINN(n_hidden_layers=3, n_neurons=64).to(device)

## 3. Funcion de perdida (Eq. 16-20)

In [ ]:
N_f = 2000
t_f = torch.rand(N_f, 1, device=device, requires_grad=True)  # colocacion en t_norm ~ U(0,1)


def compute_loss(model, lam=10.0):
    # --- Perdida fisica: R(t) = dv/dt_phys - (A - B*v), Eq. (16)-(17) ---
    v_f = model(t_f)
    dv_dtnorm = torch.autograd.grad(v_f, t_f, grad_outputs=torch.ones_like(v_f),
                                     create_graph=True, retain_graph=True)[0]
    dv_dt = dv_dtnorm / t_max  # regla de la cadena: t_norm = t_phys / t_max
    residual = dv_dt - (A - B * v_f)
    loss_physics = torch.mean(residual**2)

    # --- Perdida de datos: contra las 5 mediciones experimentales, Eq. (18) ---
    v_pred_exp = model(t_exp_t)
    loss_data = torch.mean((v_pred_exp - v_exp_t)**2)

    return loss_data + lam * loss_physics, loss_data.item(), loss_physics.item()

## 4. Entrenamiento (Adam, lr=1e-3, 5000 epocas)

In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
history = {'total': [], 'data': [], 'physics': []}

for epoch in range(5000):
    optimizer.zero_grad()
    loss, l_data, l_phys = compute_loss(model, lam=10.0)
    loss.backward()
    optimizer.step()
    history['total'].append(loss.item())
    history['data'].append(l_data)
    history['physics'].append(l_phys)
    if epoch % 500 == 0:
        print(f'epoch {epoch:5d} | loss={loss.item():.4e} | data={l_data:.4e} | physics={l_phys:.4e}')

## 5. Resultados: PINN vs. solucion analitica vs. datos experimentales (reproduce Fig. 4)

In [ ]:
t_plot_phys = np.linspace(0, t_max, 300)
t_plot_norm = torch.tensor(t_plot_phys / t_max, dtype=torch.float32, device=device).view(-1, 1)
with torch.no_grad():
    v_pinn = model(t_plot_norm).cpu().numpy().flatten()
v_analytical = v_terminal * (1 - np.exp(-B * t_plot_phys))

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].plot(t_plot_phys, v_analytical, label='Solucion analitica', linewidth=2)
axes[0].plot(t_plot_phys, v_pinn, '--', label='Prediccion PINN')
axes[0].scatter(t_exp, v_exp, color='k', zorder=5, label='Datos experimentales')
axes[0].axhline(v_terminal, color='green', linestyle=':', label=f'Velocidad terminal ({v_terminal:.3f} m/s)')
axes[0].set_xlabel('Tiempo (s)')
axes[0].set_ylabel('Velocidad (m/s)')
axes[0].set_title('Perfil de velocidad: PINN vs analitica vs experimental')
axes[0].legend()
axes[0].grid(alpha=0.3)

axes[1].semilogy(history['physics'], label='Physics loss')
axes[1].semilogy(history['data'], label='Data loss')
axes[1].semilogy(history['total'], '--', label='Total loss', alpha=0.7)
axes[1].set_xlabel('Epoca')
axes[1].set_ylabel('Loss (escala log)')
axes[1].set_title('Convergencia de la funcion de perdida (cf. Fig. 5 del paper)')
axes[1].legend()
axes[1].grid(alpha=0.3)
plt.tight_layout()
plt.show()

rmse = np.sqrt(np.mean((v_pinn - v_terminal * (1 - np.exp(-B * t_plot_phys)))**2))
print(f'RMSE PINN vs analitica (dominio completo): {rmse:.5f} m/s  (paper reporta 0.0052 m/s)')